# Datathon 2026 Round 1 — E-commerce Sales Forecasting Pipeline

Notebook này dự đoán **daily `Revenue` và `COGS` từ 2023-01-01 đến 2024-07-01** dựa trên dữ liệu lịch sử 2012-2022.

Ý tưởng chính:
- Dùng `sales.csv` làm target chính.
- Dùng các bảng phụ (`orders`, `order_items`, `products`, `payments`, `returns`, `reviews`, `inventory`, `web_traffic`, `shipments`, `promotions`) để tạo đặc trưng lịch sử.
- Tránh leakage bằng cách **không dùng thông tin cùng ngày trong tương lai**; các biến vận hành được dùng dưới dạng seasonal profile hoặc lag/rolling.
- Train nhiều mô hình tabular mạnh: **LightGBM, XGBoost, CatBoost, ExtraTrees/HistGB fallback**.
- Dự báo future bằng recursive forecasting: dự đoán từng ngày, rồi dùng dự đoán đó làm lag cho ngày sau.

## 0. Cài thư viện và cấu hình


In [ ]:
# =========================
# 0. INSTALLS + CONFIG
# =========================
import os, sys, gc, math, json, zipfile, warnings, subprocess, importlib, re
from pathlib import Path
warnings.filterwarnings("ignore")

INSTALL_EXTRA = True

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        if INSTALL_EXTRA:
            print(f"Installing {pip_name} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        else:
            print(f"[WARN] Missing package: {import_name}. Some models will be skipped.")

for imp, pipn in [
    ("lightgbm", "lightgbm"),
    ("xgboost", "xgboost"),
    ("catboost", "catboost"),
]:
    ensure_package(imp, pipn)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

try:
    from lightgbm import LGBMRegressor
except Exception as e:
    LGBMRegressor = None
    print("LightGBM unavailable:", e)

try:
    from xgboost import XGBRegressor
except Exception as e:
    XGBRegressor = None
    print("XGBoost unavailable:", e)

try:
    from catboost import CatBoostRegressor
except Exception as e:
    CatBoostRegressor = None
    print("CatBoost unavailable:", e)

SEED = 42
np.random.seed(SEED)

FAST_MODE = False
TARGETS = ["Revenue", "COGS"]

LAGS_TARGET = [1, 2, 3, 7, 14, 28, 30, 56, 90, 180, 365]
ROLL_WINDOWS_TARGET = [7, 14, 30, 56, 90, 180, 365]

LAGS_EXOG = [1, 7, 14, 30, 90]
ROLL_WINDOWS_EXOG = [7, 30, 90]

MAX_EXOG_LAG_COLS = 70 if FAST_MODE else 140
TOP_MODELS_PER_TARGET = 3
FINAL_BLEND_ML_WEIGHT = 0.90
CLIP_NEGATIVE = True
VALID_YEARS = [2022] if FAST_MODE else [2020, 2021, 2022]

print("FAST_MODE:", FAST_MODE)

## 1. Load data từ file zip


In [ ]:
# =========================
# 1. LOCATE + EXTRACT DATA
# =========================

ZIP_CANDIDATES = [
    "/content/datathon-2026-round-1.zip",
    "/mnt/data/datathon-2026-round-1.zip",
    "datathon-2026-round-1.zip",
    "./datathon-2026-round-1.zip",
]

def find_zip():
    for p in ZIP_CANDIDATES:
        if Path(p).exists():
            return Path(p)
    for p in Path(".").glob("**/*.zip"):
        if "datathon" in p.name.lower():
            return p
    return None

zip_path = find_zip()
if zip_path is None:
    raise FileNotFoundError("Không tìm thấy datathon-2026-round-1.zip. Hãy upload zip vào /content hoặc cùng thư mục notebook.")

DATA_DIR = Path("/content/datathon_data") if Path("/content").exists() else Path("./datathon_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "sales.csv").exists():
    print("Extracting:", zip_path, "->", DATA_DIR)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
else:
    print("Data already extracted:", DATA_DIR)

print("Files:")
for f in sorted(DATA_DIR.glob("*")):
    print(" -", f.name)

## 2. Đọc dữ liệu và kiểm tra schema

In [ ]:
# =========================
# 2. LOAD CSV
# =========================

DATE_COLS = {
    "sales.csv": ["Date"],
    "sample_submission.csv": ["Date"],
    "orders.csv": ["order_date"],
    "customers.csv": ["signup_date"],
    "promotions.csv": ["start_date", "end_date"],
    "returns.csv": ["return_date"],
    "reviews.csv": ["review_date"],
    "shipments.csv": ["ship_date", "delivery_date"],
    "inventory.csv": ["snapshot_date"],
    "web_traffic.csv": ["date"],
}

def read_csv_auto(name, usecols=None):
    path = DATA_DIR / name
    parse_dates = DATE_COLS.get(name, [])
    parse_dates = [c for c in parse_dates if usecols is None or c in usecols]
    return pd.read_csv(path, parse_dates=parse_dates, usecols=usecols, low_memory=False)

sales = read_csv_auto("sales.csv").sort_values("Date").reset_index(drop=True)
sample = read_csv_auto("sample_submission.csv").sort_values("Date").reset_index(drop=True)

print("sales:", sales.shape, sales["Date"].min().date(), "->", sales["Date"].max().date())
print("sample:", sample.shape, sample["Date"].min().date(), "->", sample["Date"].max().date())

schema_rows = []
for f in sorted(DATA_DIR.glob("*.csv")):
    tmp = pd.read_csv(f, nrows=5, low_memory=False)
    schema_rows.append({
        "file": f.name,
        "rows_approx": sum(1 for _ in open(f, "rb")) - 1,
        "n_cols": len(tmp.columns),
        "columns": ", ".join(tmp.columns)
    })
schema = pd.DataFrame(schema_rows)
display(schema)
display(sales.head())
display(sample.head())

## 3. EDA nhanh target

In [ ]:
# =========================
# 3. QUICK EDA
# =========================

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(sales["Date"], sales["Revenue"], lw=0.8)
axes[0].set_title("Daily Revenue")
axes[0].grid(True, alpha=0.3)

axes[1].plot(sales["Date"], sales["COGS"], lw=0.8)
axes[1].set_title("Daily COGS")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

tmp = sales.copy()
tmp["year"] = tmp["Date"].dt.year
tmp["month"] = tmp["Date"].dt.month
annual = tmp.groupby("year")[TARGETS].sum()
display(annual.tail(12))

fig, ax = plt.subplots(figsize=(12, 4))
annual["Revenue"].plot(ax=ax, marker="o", label="Revenue")
annual["COGS"].plot(ax=ax, marker="o", label="COGS")
ax.set_title("Annual totals")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 4. Tạo daily exogenous features từ tất cả bảng phụ

In [ ]:
# =========================
# 4. DAILY EXOGENOUS FEATURES
# =========================

def group_onehot_by_date(df, date_col, cat_col, prefix):
    if cat_col not in df.columns:
        return pd.DataFrame()
    temp = df[[date_col, cat_col]].dropna()
    if temp.empty:
        return pd.DataFrame()
    out = pd.crosstab(temp[date_col], temp[cat_col])
    out.columns = [f"{prefix}_{str(c)}_cnt" for c in out.columns]
    out.index.name = "Date"
    return out

def build_daily_exog():
    print("Loading tables for feature engineering...")

    orders = read_csv_auto("orders.csv")
    orders = orders.rename(columns={"order_date": "Date"})
    orders["Date"] = pd.to_datetime(orders["Date"])
    orders["zip"] = orders["zip"].astype(str)

    products = read_csv_auto("products.csv")
    items = read_csv_auto("order_items.csv")
    payments = read_csv_auto("payments.csv")
    shipments = read_csv_auto("shipments.csv")
    returns = read_csv_auto("returns.csv")
    reviews = read_csv_auto("reviews.csv")
    inventory = read_csv_auto("inventory.csv")
    web = read_csv_auto("web_traffic.csv")
    customers = read_csv_auto("customers.csv")
    geography = read_csv_auto("geography.csv")

    features = []

    od = orders.groupby("Date").agg(
        orders_cnt=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        unique_zips=("zip", "nunique"),
    )
    features.append(od)

    for col, pref in [
        ("order_status", "status"),
        ("payment_method", "order_payment"),
        ("device_type", "device"),
        ("order_source", "source"),
    ]:
        features.append(group_onehot_by_date(orders, "Date", col, pref))

    geography["zip"] = geography["zip"].astype(str)
    geo_cols = [c for c in ["zip", "region", "district"] if c in geography.columns]
    orders_geo = orders.merge(geography[geo_cols], on="zip", how="left")
    for col, pref in [("region", "region"), ("district", "district")]:
        if col in orders_geo.columns:
            oh = group_onehot_by_date(orders_geo, "Date", col, pref)
            if not oh.empty and oh.shape[1] > 30:
                keep = oh.sum().sort_values(ascending=False).head(30).index
                oh = oh[keep]
            features.append(oh)

    cust_cols = ["customer_id", "signup_date", "gender", "age_group", "acquisition_channel"]
    customers = customers[[c for c in cust_cols if c in customers.columns]].copy()
    if "signup_date" in customers.columns:
        customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
    oc = orders[["order_id", "Date", "customer_id"]].merge(customers, on="customer_id", how="left")
    if "signup_date" in oc.columns:
        oc["customer_tenure_days"] = (oc["Date"] - oc["signup_date"]).dt.days.clip(lower=0)
        features.append(oc.groupby("Date").agg(
            cust_tenure_mean=("customer_tenure_days", "mean"),
            cust_tenure_median=("customer_tenure_days", "median"),
        ))
    for col, pref in [("gender", "gender"), ("age_group", "age"), ("acquisition_channel", "acq")]:
        if col in oc.columns:
            features.append(group_onehot_by_date(oc, "Date", col, pref))

    item = items.merge(orders[["order_id", "Date"]], on="order_id", how="left")
    prod_cols = ["product_id", "category", "segment", "size", "color", "price", "cogs"]
    item = item.merge(products[[c for c in prod_cols if c in products.columns]], on="product_id", how="left")

    item["line_gross"] = item["quantity"] * item["unit_price"]
    item["line_discount"] = item["discount_amount"].fillna(0)
    item["line_net"] = (item["line_gross"] - item["line_discount"]).clip(lower=0)
    item["line_est_cogs"] = item["quantity"] * item["cogs"] if "cogs" in item.columns else np.nan
    item["line_margin"] = item["line_net"] - item["line_est_cogs"]
    item["has_promo"] = item[["promo_id", "promo_id_2"]].notna().any(axis=1).astype(int)
    item["discount_pct"] = np.where(item["line_gross"] > 0, item["line_discount"] / item["line_gross"], 0)

    item_daily = item.groupby("Date").agg(
        item_rows=("order_id", "size"),
        item_qty_sum=("quantity", "sum"),
        item_qty_mean=("quantity", "mean"),
        product_nunique=("product_id", "nunique"),
        item_gross_sum=("line_gross", "sum"),
        item_net_sum=("line_net", "sum"),
        item_discount_sum=("line_discount", "sum"),
        item_discount_mean=("line_discount", "mean"),
        item_discount_pct_mean=("discount_pct", "mean"),
        item_has_promo_mean=("has_promo", "mean"),
        item_est_cogs_sum=("line_est_cogs", "sum"),
        item_margin_sum=("line_margin", "sum"),
        item_unit_price_mean=("unit_price", "mean"),
        item_unit_price_median=("unit_price", "median"),
    )
    features.append(item_daily)

    for col in ["category", "segment"]:
        if col in item.columns:
            pvt = item.pivot_table(index="Date", columns=col, values="line_net", aggfunc="sum", fill_value=0)
            pvt.columns = [f"line_net_by_{col}_{c}" for c in pvt.columns]
            features.append(pvt)

            pvtq = item.pivot_table(index="Date", columns=col, values="quantity", aggfunc="sum", fill_value=0)
            pvtq.columns = [f"qty_by_{col}_{c}" for c in pvtq.columns]
            features.append(pvtq)

    pay = payments.merge(orders[["order_id", "Date"]], on="order_id", how="left")
    pay_daily = pay.groupby("Date").agg(
        payment_value_sum=("payment_value", "sum"),
        payment_value_mean=("payment_value", "mean"),
        payment_value_median=("payment_value", "median"),
        installments_mean=("installments", "mean"),
        installments_max=("installments", "max"),
    )
    features.append(pay_daily)
    if "payment_method" in pay.columns:
        features.append(group_onehot_by_date(pay, "Date", "payment_method", "payment"))

    ship = shipments.copy()
    ship["ship_date"] = pd.to_datetime(ship["ship_date"], errors="coerce")
    ship["delivery_date"] = pd.to_datetime(ship["delivery_date"], errors="coerce")
    ship["delivery_delay_days"] = (ship["delivery_date"] - ship["ship_date"]).dt.days
    ship_daily = ship.groupby("ship_date").agg(
        ship_cnt=("order_id", "nunique"),
        shipping_fee_sum=("shipping_fee", "sum"),
        shipping_fee_mean=("shipping_fee", "mean"),
        delivery_delay_mean=("delivery_delay_days", "mean"),
        delivery_delay_max=("delivery_delay_days", "max"),
    )
    ship_daily.index.name = "Date"
    features.append(ship_daily)

    deliv_daily = ship.groupby("delivery_date").agg(
        delivered_cnt=("order_id", "nunique"),
        delivered_delay_mean=("delivery_delay_days", "mean"),
    )
    deliv_daily.index.name = "Date"
    features.append(deliv_daily)

    ret = returns.copy()
    ret["Date"] = pd.to_datetime(ret["return_date"], errors="coerce")
    ret_daily = ret.groupby("Date").agg(
        return_cnt=("return_id", "nunique"),
        return_qty_sum=("return_quantity", "sum"),
        refund_amount_sum=("refund_amount", "sum"),
        refund_amount_mean=("refund_amount", "mean"),
    )
    features.append(ret_daily)
    if "return_reason" in ret.columns:
        features.append(group_onehot_by_date(ret, "Date", "return_reason", "return_reason"))

    rev = reviews.copy()
    rev["Date"] = pd.to_datetime(rev["review_date"], errors="coerce")
    rev_daily = rev.groupby("Date").agg(
        review_cnt=("review_id", "nunique"),
        rating_mean=("rating", "mean"),
        rating_min=("rating", "min"),
        rating_max=("rating", "max"),
        review_product_nunique=("product_id", "nunique"),
    )
    features.append(rev_daily)

    inv = inventory.copy()
    inv["snapshot_date"] = pd.to_datetime(inv["snapshot_date"], errors="coerce")
    inv_daily = inv.groupby("snapshot_date").agg(
        inv_stock_on_hand_sum=("stock_on_hand", "sum"),
        inv_stock_on_hand_mean=("stock_on_hand", "mean"),
        inv_units_received_sum=("units_received", "sum"),
        inv_units_sold_sum=("units_sold", "sum"),
        inv_stockout_days_sum=("stockout_days", "sum"),
        inv_days_of_supply_mean=("days_of_supply", "mean"),
        inv_fill_rate_mean=("fill_rate", "mean"),
        inv_stockout_flag_mean=("stockout_flag", "mean"),
        inv_overstock_flag_mean=("overstock_flag", "mean"),
        inv_reorder_flag_mean=("reorder_flag", "mean"),
        inv_sell_through_rate_mean=("sell_through_rate", "mean"),
    )
    inv_daily.index.name = "Date"
    features.append(inv_daily)

    web = web.rename(columns={"date": "Date"})
    web["Date"] = pd.to_datetime(web["Date"], errors="coerce")
    web_daily = web.groupby("Date").agg(
        web_sessions=("sessions", "sum"),
        web_unique_visitors=("unique_visitors", "sum"),
        web_page_views=("page_views", "sum"),
        web_bounce_rate_mean=("bounce_rate", "mean"),
        web_avg_session_duration_mean=("avg_session_duration_sec", "mean"),
    )
    features.append(web_daily)
    if "traffic_source" in web.columns:
        features.append(group_onehot_by_date(web, "Date", "traffic_source", "traffic"))

    promos = read_csv_auto("promotions.csv")
    promo_daily_rows = []
    if len(promos):
        min_date = sales["Date"].min()
        max_date = sample["Date"].max()
        for _, r in promos.iterrows():
            st = pd.to_datetime(r.get("start_date"), errors="coerce")
            en = pd.to_datetime(r.get("end_date"), errors="coerce")
            if pd.isna(st) or pd.isna(en):
                continue
            dates = pd.date_range(max(st, min_date), min(en, max_date), freq="D")
            for d in dates:
                promo_daily_rows.append({
                    "Date": d,
                    "promo_active_cnt": 1,
                    "promo_discount_value_sum": r.get("discount_value", 0) if pd.notna(r.get("discount_value", 0)) else 0,
                    "promo_stackable_sum": r.get("stackable_flag", 0) if pd.notna(r.get("stackable_flag", 0)) else 0,
                    "promo_min_order_value_mean": r.get("min_order_value", 0) if pd.notna(r.get("min_order_value", 0)) else 0,
                })
    if promo_daily_rows:
        promo_daily = pd.DataFrame(promo_daily_rows).groupby("Date").agg(
            promo_active_cnt=("promo_active_cnt", "sum"),
            promo_discount_value_sum=("promo_discount_value_sum", "sum"),
            promo_stackable_sum=("promo_stackable_sum", "sum"),
            promo_min_order_value_mean=("promo_min_order_value_mean", "mean"),
        )
        features.append(promo_daily)

    daily_exog = pd.concat([x for x in features if x is not None and not x.empty], axis=1).sort_index()
    daily_exog = daily_exog[~daily_exog.index.isna()]
    daily_exog = daily_exog.groupby(level=0).sum(min_count=1)

    all_dates = pd.date_range(sales["Date"].min(), sample["Date"].max(), freq="D")
    daily_exog = daily_exog.reindex(all_dates)
    daily_exog.index.name = "Date"
    daily_exog = daily_exog.apply(pd.to_numeric, errors="coerce")
    daily_exog = daily_exog.replace([np.inf, -np.inf], np.nan)

    print("daily_exog:", daily_exog.shape)
    return daily_exog

daily_exog = build_daily_exog()
display(daily_exog.head())
display(daily_exog.tail())
gc.collect()

## 5. Seasonal baseline

In [ ]:
# =========================
# 5. SEASONAL BASELINE
# =========================

def seasonal_trend_forecast(train_sales, future_dates, targets=TARGETS):
    train = train_sales.copy().sort_values("Date")
    train["year"] = train["Date"].dt.year
    train["month"] = train["Date"].dt.month
    train["day"] = train["Date"].dt.day

    annual = train.groupby("year")[list(targets)].sum()
    full_years = annual.loc[annual.index >= 2013].copy()

    growth = {}
    for t in targets:
        yoy = full_years[t].pct_change().replace([np.inf, -np.inf], np.nan).dropna()
        growth[t] = float((1 + yoy).prod() ** (1 / len(yoy))) if len(yoy) else 1.0

    tmp = train.copy()
    for t in targets:
        yearly_mean = tmp.groupby("year")[t].transform("mean")
        tmp[f"{t}_norm"] = tmp[t] / yearly_mean.replace(0, np.nan)

    prof = tmp.groupby(["month", "day"])[[f"{t}_norm" for t in targets]].mean().reset_index()

    last_full_year = train["year"].max()
    base_level = {}
    for t in targets:
        days = 366 if pd.Timestamp(f"{last_full_year}-12-31").is_leap_year else 365
        base_level[t] = annual.loc[last_full_year, t] / days if last_full_year in annual.index else train[t].mean()

    fut = pd.DataFrame({"Date": pd.to_datetime(future_dates)})
    fut["year"] = fut["Date"].dt.year
    fut["month"] = fut["Date"].dt.month
    fut["day"] = fut["Date"].dt.day
    fut["years_ahead"] = fut["year"] - last_full_year
    fut = fut.merge(prof, on=["month", "day"], how="left")

    out = pd.DataFrame({"Date": fut["Date"]})
    for t in targets:
        norm_col = f"{t}_norm"
        fut[norm_col] = fut[norm_col].fillna(1.0)
        out[t] = base_level[t] * (growth[t] ** fut["years_ahead"]) * fut[norm_col]
    return out

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    den = np.abs(y_true) + np.abs(y_pred)
    return np.mean(np.where(den == 0, 0, 2 * np.abs(y_pred - y_true) / den)) * 100

def metrics_df(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "SMAPE": smape(y_true, y_pred),
    }

cutoff = pd.Timestamp("2022-01-01")
base_valid = seasonal_trend_forecast(
    sales[sales["Date"] < cutoff],
    sales[sales["Date"] >= cutoff]["Date"],
)
valid_true = sales[sales["Date"] >= cutoff][["Date"] + TARGETS].merge(base_valid, on="Date", suffixes=("_true", "_pred"))

baseline_metrics = []
for t in TARGETS:
    m = metrics_df(valid_true[f"{t}_true"], valid_true[f"{t}_pred"])
    m["target"] = t
    m["model"] = "seasonal_trend_baseline"
    baseline_metrics.append(m)
baseline_metrics = pd.DataFrame(baseline_metrics)
display(baseline_metrics)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(valid_true["Date"], valid_true["Revenue_true"], label="Revenue true", lw=0.8)
ax.plot(valid_true["Date"], valid_true["Revenue_pred"], label="Revenue baseline", lw=0.8)
ax.legend(); ax.grid(True, alpha=0.3); ax.set_title("Baseline validation on 2022")
plt.show()

## 6. Feature engineering supervised
Tạo calendar, seasonal profile, lag, rolling. Các target lag đều `shift(1)` trở lên.

In [ ]:
# =========================
# 6. SUPERVISED FEATURE ENGINEERING
# =========================


def sanitize_feature_name(name):
    if name == "Date":
        return name
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name))
    name = re.sub(r"_+", "_", name).strip("_")
    if not name:
        name = "feature"
    if name[0].isdigit():
        name = "f_" + name
    return name

def sanitize_df_columns(df):
    new_cols = []
    seen = {}
    for c in df.columns:
        base = sanitize_feature_name(c)
        if base in seen:
            seen[base] += 1
            base = f"{base}_{seen[base]}"
        else:
            seen[base] = 0
        new_cols.append(base)
    df = df.copy()
    df.columns = new_cols
    return df


def add_calendar_features(date_series):
    d = pd.to_datetime(date_series)
    out = pd.DataFrame(index=d.index)
    out["year"] = d.dt.year
    out["month"] = d.dt.month
    out["day"] = d.dt.day
    out["dayofweek"] = d.dt.dayofweek
    out["dayofyear"] = d.dt.dayofyear
    out["weekofyear"] = d.dt.isocalendar().week.astype(int)
    out["quarter"] = d.dt.quarter
    out["is_weekend"] = (d.dt.dayofweek >= 5).astype(int)
    out["is_month_start"] = d.dt.is_month_start.astype(int)
    out["is_month_end"] = d.dt.is_month_end.astype(int)
    out["is_quarter_start"] = d.dt.is_quarter_start.astype(int)
    out["is_quarter_end"] = d.dt.is_quarter_end.astype(int)
    out["is_year_start"] = d.dt.is_year_start.astype(int)
    out["is_year_end"] = d.dt.is_year_end.astype(int)

    for period, name in [(7, "week"), (12, "month"), (365.25, "year")]:
        x = d.dt.dayofweek if name == "week" else (d.dt.month if name == "month" else d.dt.dayofyear)
        out[f"{name}_sin"] = np.sin(2 * np.pi * x / period)
        out[f"{name}_cos"] = np.cos(2 * np.pi * x / period)

    fixed_md = set([(1, 1), (4, 30), (5, 1), (9, 2), (12, 24), (12, 25), (12, 31)])
    out["is_fixed_holiday"] = [(m, day) in fixed_md for m, day in zip(d.dt.month, d.dt.day)]
    out["is_fixed_holiday"] = out["is_fixed_holiday"].astype(int)
    return out

def make_profile_features(profile_df, all_dates, cols, prefix):
    cols = [c for c in cols if c in profile_df.columns]
    dates = pd.DataFrame({"Date": pd.to_datetime(all_dates)})
    out = dates.copy()
    out["dayofyear"] = out["Date"].dt.dayofyear
    out["month"] = out["Date"].dt.month
    out["dayofweek"] = out["Date"].dt.dayofweek

    if len(cols) == 0 or profile_df.empty:
        return pd.DataFrame(index=dates.index)

    src = profile_df[["Date"] + cols].copy()
    src["Date"] = pd.to_datetime(src["Date"])
    src["dayofyear"] = src["Date"].dt.dayofyear
    src["month"] = src["Date"].dt.month
    src["dayofweek"] = src["Date"].dt.dayofweek

    for c in cols:
        src[c] = pd.to_numeric(src[c], errors="coerce")

    doy = src.groupby("dayofyear")[cols].mean().add_prefix(f"{prefix}_doy_").reset_index()
    mdw = src.groupby(["month", "dayofweek"])[cols].mean().add_prefix(f"{prefix}_mdw_").reset_index()
    mon = src.groupby("month")[cols].mean().add_prefix(f"{prefix}_month_").reset_index()

    out = out.merge(doy, on="dayofyear", how="left")
    out = out.merge(mdw, on=["month", "dayofweek"], how="left")
    out = out.merge(mon, on="month", how="left")

    feat = out.drop(columns=["Date", "dayofyear", "month", "dayofweek"])
    for c in feat.columns:
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median())
    return feat.reset_index(drop=True)

def select_exog_columns(daily_exog, history_dates, max_cols=MAX_EXOG_LAG_COLS):
    hist = daily_exog.loc[daily_exog.index.isin(history_dates)].copy()
    hist = hist.replace([np.inf, -np.inf], np.nan)
    variances = hist.var(numeric_only=True).sort_values(ascending=False)
    variances = variances[variances > 0]
    return list(variances.head(max_cols).index)

all_dates = pd.date_range(sales["Date"].min(), sample["Date"].max(), freq="D")
base_df = pd.DataFrame({"Date": all_dates})
base_df = base_df.merge(sales, on="Date", how="left")
base_df = base_df.merge(daily_exog.reset_index(), on="Date", how="left")

history_dates = set(sales["Date"])
selected_exog_cols = select_exog_columns(daily_exog, history_dates, MAX_EXOG_LAG_COLS)
print("Selected exog columns:", len(selected_exog_cols))
print(selected_exog_cols[:30])

def add_estimated_future_exog(base_df, selected_exog_cols, history_end_date):
    df = base_df.copy()
    hist_mask = df["Date"] <= history_end_date
    profile_src = df.loc[hist_mask, ["Date"] + selected_exog_cols].copy()
    prof = make_profile_features(profile_src, df["Date"], selected_exog_cols, "raw_exog_profile")
    for c in selected_exog_cols:
        candidate_cols = [x for x in prof.columns if x.endswith("_" + c)]
        if not candidate_cols:
            continue
        ordered = [x for x in candidate_cols if "_doy_" in x] + [x for x in candidate_cols if "_mdw_" in x] + [x for x in candidate_cols if "_month_" in x]
        proxy = None
        for cc in ordered:
            proxy = prof[cc].copy() if proxy is None else proxy.fillna(prof[cc])
        if proxy is not None:
            mask = (~hist_mask) & (df[c].isna())
            df.loc[mask, c] = proxy.loc[mask].values
    return df

base_df = add_estimated_future_exog(base_df, selected_exog_cols, sales["Date"].max())

def make_supervised_features(work_df, profile_source, selected_exog_cols):
    df = work_df.sort_values("Date").reset_index(drop=True).copy()
    dates = df["Date"]

    df["gross_margin"] = df["Revenue"] - df["COGS"]
    df["cogs_ratio"] = np.where(df["Revenue"] > 0, df["COGS"] / df["Revenue"], np.nan)

    profile_source = profile_source.sort_values("Date").copy()
    profile_source["gross_margin"] = profile_source["Revenue"] - profile_source["COGS"]
    profile_source["cogs_ratio"] = np.where(profile_source["Revenue"] > 0, profile_source["COGS"] / profile_source["Revenue"], np.nan)

    feat = add_calendar_features(dates)

    target_profile_cols = TARGETS + ["gross_margin", "cogs_ratio"]
    prof_t = make_profile_features(profile_source, dates, target_profile_cols, "prof_target")
    prof_t.index = feat.index
    feat = pd.concat([feat, prof_t], axis=1)

    exog_source = profile_source[["Date"] + [c for c in selected_exog_cols if c in profile_source.columns]].copy()
    prof_x = make_profile_features(exog_source, dates, selected_exog_cols, "prof_exog")
    prof_x.index = feat.index
    feat = pd.concat([feat, prof_x], axis=1)

    lag_target_cols = TARGETS + ["gross_margin", "cogs_ratio"]
    for c in lag_target_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        for lag in LAGS_TARGET:
            feat[f"{c}_lag_{lag}"] = s.shift(lag)
        s1 = s.shift(1)
        for w in ROLL_WINDOWS_TARGET:
            feat[f"{c}_roll_mean_{w}"] = s1.rolling(w, min_periods=max(2, w // 4)).mean()
            feat[f"{c}_roll_std_{w}"] = s1.rolling(w, min_periods=max(2, w // 4)).std()
            feat[f"{c}_roll_min_{w}"] = s1.rolling(w, min_periods=max(2, w // 4)).min()
            feat[f"{c}_roll_max_{w}"] = s1.rolling(w, min_periods=max(2, w // 4)).max()

    for c in selected_exog_cols:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        for lag in LAGS_EXOG:
            feat[f"{c}_lag_{lag}"] = s.shift(lag)
        s1 = s.shift(1)
        for w in ROLL_WINDOWS_EXOG:
            feat[f"{c}_roll_mean_{w}"] = s1.rolling(w, min_periods=max(2, w // 3)).mean()

    start_date = df["Date"].min()
    feat["days_from_start"] = (df["Date"] - start_date).dt.days
    feat["years_from_start"] = feat["days_from_start"] / 365.25

    feat = feat.replace([np.inf, -np.inf], np.nan)
    feat["Date"] = dates.values
    feat = sanitize_df_columns(feat)
    return feat

profile_source_full = base_df[base_df["Date"] <= sales["Date"].max()].copy()
features_all = make_supervised_features(base_df, profile_source_full, selected_exog_cols)

print("features_all:", features_all.shape)
display(features_all.head())

Selected exog columns: 140
['item_gross_sum', 'item_net_sum', 'payment_value_sum', 'item_est_cogs_sum', 'line_net_by_category_Streetwear', 'line_net_by_segment_Everyday', 'item_margin_sum', 'line_net_by_segment_Balanced', 'line_net_by_segment_Performance', 'line_net_by_category_Outdoor', 'line_net_by_segment_Activewear', 'item_discount_sum', 'line_net_by_category_Casual', 'line_net_by_segment_Premium', 'line_net_by_segment_All-weather', 'refund_amount_sum', 'line_net_by_category_GenZ', 'line_net_by_segment_Trendy', 'promo_min_order_value_mean', 'line_net_by_segment_Standard', 'web_page_views', 'inv_stock_on_hand_sum', 'web_sessions', 'web_unique_visitors', 'refund_amount_mean', 'payment_value_mean', 'payment_value_median', 'inv_units_received_sum', 'inv_units_sold_sum', 'item_discount_mean']
features_all: (4381, 1732)


,year,month,day,dayofweek,dayofyear,weekofyear,quarter,is_weekend,is_month_start,is_month_end,...,return_reason_changed_mind_cnt_lag_7,return_reason_changed_mind_cnt_lag_14,return_reason_changed_mind_cnt_lag_30,return_reason_changed_mind_cnt_lag_90,return_reason_changed_mind_cnt_roll_mean_7,return_reason_changed_mind_cnt_roll_mean_30,return_reason_changed_mind_cnt_roll_mean_90,days_from_start,years_from_start,Date
0,2012,7,4,2,186,27,3,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.000000,2012-07-04
1,2012,7,5,3,187,27,3,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.002738,2012-07-05
2,2012,7,6,4,188,27,3,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.005476,2012-07-06
3,2012,7,7,5,189,27,3,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0.008214,2012-07-07
4,2012,7,8,6,190,27,3,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0.010951,2012-07-08


## 7. Train/CV nhiều model

In [ ]:
# =========================
# 7. MODEL TRAINING + CV
# =========================

def get_model_specs():
    specs = {}

    if LGBMRegressor is not None:
        specs["lgbm"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(
                n_estimators=900 if FAST_MODE else 1800,
                learning_rate=0.025 if FAST_MODE else 0.015,
                num_leaves=64 if FAST_MODE else 96,
                subsample=0.85,
                colsample_bytree=0.80,
                reg_alpha=0.15,
                reg_lambda=1.5,
                random_state=SEED,
                objective="regression",
                n_jobs=-1,
                verbose=-1
            ))
        ])

    if XGBRegressor is not None:
        specs["xgb"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=700 if FAST_MODE else 1300,
                max_depth=4 if FAST_MODE else 5,
                learning_rate=0.025 if FAST_MODE else 0.015,
                subsample=0.85,
                colsample_bytree=0.80,
                reg_alpha=0.05,
                reg_lambda=1.5,
                min_child_weight=5,
                objective="reg:squarederror",
                random_state=SEED,
                n_jobs=-1,
                tree_method="hist",
                verbosity=0
            ))
        ])

    if CatBoostRegressor is not None:
        specs["catboost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=700 if FAST_MODE else 1400,
                depth=6,
                learning_rate=0.035 if FAST_MODE else 0.02,
                loss_function="RMSE",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            ))
        ])

    specs["extratrees"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=400 if FAST_MODE else 800,
            max_depth=None,
            min_samples_leaf=2,
            random_state=SEED,
            n_jobs=-1
        ))
    ])

    specs["hgb"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=500 if FAST_MODE else 1000,
            learning_rate=0.035,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=SEED
        ))
    ])

    specs["ridge"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=50.0, random_state=SEED))
    ])

    return specs

def make_sample_weight(dates, y):
    dates = pd.to_datetime(dates)
    year = dates.dt.year
    recency = 1.0 + 0.15 * (year - year.min())
    recency = recency / np.nanmean(recency)
    y = np.asarray(y, dtype=float)
    q90 = np.nanquantile(y, 0.90)
    q10 = np.nanquantile(y, 0.10)
    extreme = np.where((y >= q90) | (y <= q10), 1.25, 1.0)
    return np.asarray(recency * extreme, dtype=float)

def fit_log_model(model, X, y, dates=None):
    y_log = np.log1p(np.clip(y, 0, None))
    if dates is not None:
        w = make_sample_weight(pd.Series(dates), y)
        try:
            model.fit(X, y_log, model__sample_weight=w)
        except Exception:
            model.fit(X, y_log)
    else:
        model.fit(X, y_log)
    return model

def predict_log_model(model, X):
    pred = np.expm1(model.predict(X))
    return np.clip(pred, 0, None) if CLIP_NEGATIVE else pred

def evaluate_prediction(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred),
        "SMAPE": smape(y_true, pred),
    }

def get_feature_columns(feat_df):
    return [c for c in feat_df.columns if c != "Date" and pd.api.types.is_numeric_dtype(feat_df[c])]

model_specs = get_model_specs()
print("Models:", list(model_specs.keys()))

cv_rows = []
cv_predictions = {}

for valid_year in VALID_YEARS:
    val_start = pd.Timestamp(f"{valid_year}-01-01")
    val_end = pd.Timestamp(f"{valid_year}-12-31")

    profile_source = base_df[base_df["Date"] < val_start].copy()
    feat = make_supervised_features(base_df, profile_source, selected_exog_cols)
    feat_cols = get_feature_columns(feat)

    known_dates = set(sales["Date"])
    train_mask = (feat["Date"] < val_start) & (feat["Date"].isin(known_dates))
    val_mask = (feat["Date"] >= val_start) & (feat["Date"] <= val_end) & (feat["Date"].isin(known_dates))

    train_part = feat.loc[train_mask].copy()
    val_part = feat.loc[val_mask].copy()

    y_table = base_df[["Date"] + TARGETS].copy()
    train_part = train_part.merge(y_table, on="Date", how="left")
    val_part = val_part.merge(y_table, on="Date", how="left")

    train_part = train_part.dropna(subset=["Revenue_lag_365", "COGS_lag_365"])
    val_part = val_part.dropna(subset=["Revenue_lag_1", "COGS_lag_1"])

    X_train = train_part[feat_cols]
    X_val = val_part[feat_cols]
    print(f"\nValidation year {valid_year}: train={X_train.shape}, val={X_val.shape}")

    for target in TARGETS:
        y_train = train_part[target].values
        y_val = val_part[target].values

        for name, spec in model_specs.items():
            model = clone(spec)
            try:
                model = fit_log_model(model, X_train, y_train, dates=train_part["Date"])
                pred = predict_log_model(model, X_val)
                m = evaluate_prediction(y_val, pred)
                m.update({"valid_year": valid_year, "target": target, "model": name})
                cv_rows.append(m)
                cv_predictions[(valid_year, target, name)] = pd.DataFrame({
                    "Date": val_part["Date"].values,
                    "y_true": y_val,
                    "pred": pred,
                })
                print(f"  {target:7s} | {name:10s} | RMSE={m['RMSE']:,.2f} | MAE={m['MAE']:,.2f} | SMAPE={m['SMAPE']:.2f}")
            except Exception as e:
                print(f"  [SKIP] {target} {name}: {e}")

cv_results = pd.DataFrame(cv_rows)
display(cv_results.sort_values(["target", "RMSE"]).reset_index(drop=True))

cv_summary = (
    cv_results
    .groupby(["target", "model"], as_index=False)
    .agg(RMSE=("RMSE", "mean"), MAE=("MAE", "mean"), SMAPE=("SMAPE", "mean"), R2=("R2", "mean"))
    .sort_values(["target", "RMSE"])
)
display(cv_summary)

Models: ['lgbm', 'xgb', 'catboost', 'extratrees', 'hgb', 'ridge']

Validation year 2020: train=(2372, 1731), val=(366, 1731)
  Revenue | lgbm       | RMSE=761,949.81 | MAE=523,700.05 | SMAPE=19.78
  Revenue | xgb        | RMSE=738,928.17 | MAE=515,058.83 | SMAPE=19.61
  Revenue | catboost   | RMSE=756,711.82 | MAE=525,880.42 | SMAPE=19.71
  Revenue | extratrees | RMSE=712,586.90 | MAE=509,371.35 | SMAPE=19.44
  Revenue | hgb        | RMSE=739,712.18 | MAE=504,074.81 | SMAPE=19.16
  Revenue | ridge      | RMSE=972,663.84 | MAE=693,556.50 | SMAPE=26.20
  COGS    | lgbm       | RMSE=603,508.78 | MAE=421,377.25 | SMAPE=18.69
  COGS    | xgb        | RMSE=597,945.43 | MAE=416,766.38 | SMAPE=18.93
  COGS    | catboost   | RMSE=618,451.40 | MAE=430,450.32 | SMAPE=19.27
  COGS    | extratrees | RMSE=575,726.65 | MAE=404,115.55 | SMAPE=18.60
  COGS    | hgb        | RMSE=605,843.19 | MAE=419,140.16 | SMAPE=18.79
  COGS    | ridge      | RMSE=826,652.38 | MAE=587,863.61 | SMAPE=26.47

Validation

## 8. Train full history và dự báo recursive 2023-2024

In [ ]:
# =========================
# 8. TRAIN FULL + RECURSIVE FORECAST
# =========================

best_models_by_target = {}
for target in TARGETS:
    sub = cv_summary[cv_summary["target"] == target].sort_values("RMSE")
    top_names = sub["model"].head(TOP_MODELS_PER_TARGET).tolist()
    if len(top_names) == 0:
        top_names = list(model_specs.keys())[:TOP_MODELS_PER_TARGET]
    best_models_by_target[target] = top_names
    print(f"{target}: top models = {top_names}")

history_end = sales["Date"].max()
profile_source_full = base_df[base_df["Date"] <= history_end].copy()
features_full = make_supervised_features(base_df, profile_source_full, selected_exog_cols)
feature_cols = get_feature_columns(features_full)

train_full = features_full[features_full["Date"].isin(set(sales["Date"]))].copy()
train_full = train_full.merge(base_df[["Date"] + TARGETS], on="Date", how="left")
train_full = train_full.dropna(subset=["Revenue_lag_365", "COGS_lag_365"])

X_full = train_full[feature_cols]
print("Full train:", X_full.shape, "features:", len(feature_cols))

trained_models = {t: [] for t in TARGETS}

for target in TARGETS:
    y_full = train_full[target].values
    top_names = best_models_by_target[target]
    target_cv = cv_summary[cv_summary["target"] == target].set_index("model")
    rmses = []
    for name in top_names:
        rmse = target_cv.loc[name, "RMSE"] if name in target_cv.index else np.nan
        rmses.append(rmse)
    inv = np.array([1 / r if np.isfinite(r) and r > 0 else 1.0 for r in rmses])
    weights = inv / inv.sum()

    print(f"\nTraining final models for {target}")
    for name, weight in zip(top_names, weights):
        model = clone(model_specs[name])
        model = fit_log_model(model, X_full, y_full, dates=train_full["Date"])
        trained_models[target].append((name, weight, model))
        print(f"  {name:10s} weight={weight:.3f}")

def ensemble_predict(target, X_row):
    preds = []
    weights = []
    for name, w, model in trained_models[target]:
        p = predict_log_model(model, X_row)[0]
        preds.append(p)
        weights.append(w)
    return float(np.average(preds, weights=weights))

future_dates = pd.to_datetime(sample["Date"]).sort_values()
baseline_future = seasonal_trend_forecast(sales, future_dates).set_index("Date")

work = base_df.copy().sort_values("Date").reset_index(drop=True)
future_dates_list = list(future_dates)

for i, d in enumerate(future_dates_list, 1):
    feat_now = make_supervised_features(work, profile_source_full, selected_exog_cols)
    row = feat_now[feat_now["Date"] == d]
    if row.empty:
        raise RuntimeError(f"Missing feature row for {d}")
    X_row = row[feature_cols]

    day_preds = {}
    for target in TARGETS:
        ml_pred = ensemble_predict(target, X_row)
        base_pred = float(baseline_future.loc[d, target]) if d in baseline_future.index else ml_pred
        final_pred = FINAL_BLEND_ML_WEIGHT * ml_pred + (1 - FINAL_BLEND_ML_WEIGHT) * base_pred
        if CLIP_NEGATIVE:
            final_pred = max(0.0, final_pred)
        day_preds[target] = final_pred

    idx = work.index[work["Date"] == d][0]
    for target in TARGETS:
        work.loc[idx, target] = day_preds[target]

    if i % 100 == 0 or i == 1 or i == len(future_dates_list):
        print(f"Predicted {i}/{len(future_dates_list)}: {d.date()} -> Revenue={day_preds['Revenue']:,.0f}, COGS={day_preds['COGS']:,.0f}")

submission = work[work["Date"].isin(set(future_dates))][["Date"] + TARGETS].copy()
submission["Date"] = submission["Date"].dt.strftime("%Y-%m-%d")
submission[TARGETS] = submission[TARGETS].round(2)
submission["Revenue"] = submission["Revenue"].clip(lower=0)
submission["COGS"] = submission["COGS"].clip(lower=0)

OUT_PATH = DATA_DIR / "submission_pipeline.csv"
submission.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
display(submission.head())
display(submission.tail())

## 9. So sánh forecast với baseline và visualize

In [ ]:
# =========================
# 9. VISUALIZE FUTURE FORECAST
# =========================

sub_plot = submission.copy()
sub_plot["Date"] = pd.to_datetime(sub_plot["Date"])
base_plot = baseline_future.reset_index()

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for ax, target in zip(axes, TARGETS):
    ax.plot(sales["Date"], sales[target], lw=0.6, label=f"history {target}")
    ax.plot(sub_plot["Date"], sub_plot[target], lw=1.0, label=f"ML recursive {target}")
    ax.plot(base_plot["Date"], base_plot[target], lw=0.8, linestyle="--", label=f"baseline {target}")
    ax.set_title(target)
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

for target in TARGETS:
    print(target)
    print("History:")
    print(sales[target].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).to_string())
    print("Forecast:")
    print(sub_plot[target].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).to_string())
    print()

## 10. Lưu report và feature importance

In [ ]:
# =========================
# 10. REPORT + FEATURE IMPORTANCE
# =========================

report = {
    "FAST_MODE": FAST_MODE,
    "valid_years": VALID_YEARS,
    "n_features": len(feature_cols),
    "selected_exog_cols": selected_exog_cols,
    "best_models_by_target": best_models_by_target,
    "final_blend_ml_weight": FINAL_BLEND_ML_WEIGHT,
}

with open(DATA_DIR / "pipeline_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

cv_results.to_csv(DATA_DIR / "cv_results_detailed.csv", index=False)
cv_summary.to_csv(DATA_DIR / "cv_summary.csv", index=False)

print("Saved report files:")
print(" -", DATA_DIR / "pipeline_report.json")
print(" -", DATA_DIR / "cv_results_detailed.csv")
print(" -", DATA_DIR / "cv_summary.csv")
print(" -", OUT_PATH)

for target in TARGETS:
    print("\nFeature importance for", target)
    found = False
    for name, w, pipe in trained_models[target]:
        model = pipe.named_steps.get("model")
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
            # Check if the length of feature_cols matches the length of importances
            if len(feature_cols) == len(importances):
                imp = pd.DataFrame({
                    "feature": feature_cols,
                    "importance": importances
                }).sort_values("importance", ascending=False).head(40)
                display(imp)
                found = True
                break
            else:
                print(f"  Warning: Skipping feature importance display for {name} (target: {target}) due to length mismatch.")
                print(f"  Expected {len(feature_cols)} features, but model returned {len(importances)} importances.")
    if not found:
        print("No feature_importances_ available or length mismatch for selected models.")